
### Model A — Euclidean Distance (Baseline)

- Standardize player features using `StandardScaler`.
- Compute Euclidean distances between players of the same tactical role.
- Return the K nearest players based on their statistical profiles.
- Uses all original features without dimensionality reduction.


In [28]:
import os
import sys
from pathlib import Path

# Locate the project root
project_root = Path.cwd()

if not (project_root / "src").is_dir():
    project_root = project_root.parent

# Set the project root as the working directory
os.chdir(project_root)
sys.path.insert(0, str(project_root))

In [29]:

import pandas as pd
from src.similarity import build_role_matrices, find_similar_players

df = pd.read_parquet("data/processed/player_features.parquet")
matrices = build_role_matrices()

targets = ["Kanté", "alli", "kane"]

for name in targets:
    matches = df[
        df["player_name"].str.contains(name, case=False, na=False)
        & df["eligible_for_similarity"]
    ]

    for _, player in matches.iterrows():
        results = find_similar_players(
            player_id=int(player["player_id"]),
            matrices=matrices,
            top_k=5,
        )

        print(f"\n{player['player_name']} ({player['role']})")
        print(results[["player_name", "distance"]].to_string(index=False))



N''Golo Kanté (defensive_midfielder)
            player_name  distance
Mousa Sidi Yaya Dembélé  3.658275
           Leon Britton  3.903032
         Victor Wanyama  3.921062
         James McArthur  3.944850
          Nemanja Matić  4.453274

Chris Smalling (centre_back)
                   player_name  distance
                Micah Richards  3.068939
      Martín Gastón Demichelis  3.419181
              Eliaquim Mangala  3.491570
Sébastien Aymar Bassong Nguena  3.640253
                  Ciaran Clark  3.669796

Bamidele Alli (attacking_midfielder)
                        player_name  distance
Roberto Firmino Barbosa de Oliveira  2.809326
                 Adam David Lallana  3.141173
                    Steven Naismith  3.251925
                  Bojan Krkíc Pérez  3.856258
    Oscar dos Santos Emboaba Júnior  4.393315

Harry Kane (forward)
                   player_name  distance
             Odion Jude Ighalo  2.417352
           Aleksandar Mitrović  2.499173
          Romelu Lukaku

In [30]:

import numpy as np
import pandas as pd

from src.similarity import build_role_matrices

matrices = build_role_matrices()


def get_neighbors(X, k=5):
    # Compute distances between all players
    values = X.to_numpy()
    distances = np.linalg.norm(
        values[:, None, :] - values[None, :, :],
        axis=2,
    )

    # Exclude each player from their own neighbors
    np.fill_diagonal(distances, np.inf)

    # Return the indices of the k nearest players
    return np.argsort(distances, axis=1)[:, :k]


def evaluate_stability(matrices, k=5):
    results = []

    for role, data in matrices.items():
        X = data["X"]

        # Original recommendations
        baseline = get_neighbors(X, k)

        feature_scores = {}

        for feature in X.columns:
            # Remove one feature and recalculate neighbors
            X_reduced = X.drop(columns=[feature])
            new_neighbors = get_neighbors(X_reduced, k)

            # Compare original and new top-k recommendations
            overlaps = [
                len(set(a) & set(b)) / k
                for a, b in zip(baseline, new_neighbors)
            ]

            feature_scores[feature] = np.mean(overlaps)

        results.append({
            "role": role,
            "players": len(X),
            "mean_stability": np.mean(list(feature_scores.values())),
            "most_sensitive_feature": min(feature_scores, key=feature_scores.get),
        })

    return pd.DataFrame(results)


stability = evaluate_stability(matrices)
display(stability.round(3))


,role,players,mean_stability,most_sensitive_feature
0,goalkeeper,27,0.806,passes_attempted_per90
1,centre_back,60,0.827,interceptions_per90
2,full_back,59,0.829,interceptions_per90
3,defensive_midfielder,37,0.858,ball_recoveries_per90
4,central_midfielder,31,0.876,dribble_completion_rate
5,wide_player,60,0.860,average_non_penalty_xg_per_shot
6,attacking_midfielder,17,0.913,average_carry_length
7,forward,36,0.847,average_non_penalty_xg_per_shot


In [31]:

import numpy as np

# Compare Wes Morgan and Robert Huth
X = matrices["centre_back"]["X"]

morgan = X.loc[3813]
huth = X.loc[40123]

# Absolute differences between standardized features
differences = (morgan - huth).abs().sort_values()

print("Most similar features:")
display(differences.head(5))

print("Most different features:")
display(differences.tail(5).sort_values(ascending=False))


Most similar features:


pressures_per90           0.003540
passes_attempted_per90    0.029878
pass_completion_rate      0.118268
ball_recoveries_per90     0.158555
interceptions_per90       0.210320
dtype: float64

Most different features:


clearances_per90                      1.822297
blocks_per90                          1.119785
average_completed_pass_progression    1.005818
progressive_carries_per90             0.560824
average_completed_pass_length         0.398081
dtype: float64

### Model B — PCA + NearestNeighbors

- Standardize player features using `StandardScaler`.
- Apply PCA to retain at least 90% of the explained variance.
- Use `NearestNeighbors` (scikit-learn) with Euclidean distance.
- Return the K nearest players in the reduced feature space.

In [32]:

import pandas as pd
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from src.similarity import find_similar_players

# Select a player and his role matrix
player_id = 3813  # Wes Morgan
data = matrices["centre_back"]
X = data["X"]

# Learn a lower-dimensional representation
pca = PCA(n_components=0.90, svd_solver="full")
X_pca = pca.fit_transform(X)

print("Original dimensions:", X.shape[1])
print("PCA dimensions:", X_pca.shape[1])
print("Explained variance:", round(pca.explained_variance_ratio_.sum(), 3))

# Fit a nearest-neighbor search on PCA features
model = NearestNeighbors(metric="euclidean")
model.fit(X_pca)

# Find Wes Morgan's 5 nearest neighbors
position = X.index.get_loc(player_id)

distances, indices = model.kneighbors(
    X_pca[position:position + 1],
    n_neighbors=6,
)

neighbors = pd.DataFrame({
    "player_id": X.index[indices[0]],
    "distance": distances[0],
})

# Exclude Wes Morgan himself
neighbors = neighbors[
    neighbors["player_id"] != player_id
].head(5)

neighbors["player_name"] = data["players"].loc[
    neighbors["player_id"], "player_name"
].to_numpy()

# Compare the two methods
baseline = find_similar_players(player_id, matrices, top_k=5)

print("\nOriginal model:")
display(baseline[["player_name", "distance"]])

print("PCA + NearestNeighbors:")
display(neighbors[["player_name", "distance"]])

overlap = len(
    set(baseline["player_id"]) & set(neighbors["player_id"])
) / 5

print(f"Top-5 overlap: {overlap:.0%}")


Original dimensions: 11
PCA dimensions: 8
Explained variance: 0.937

Original model:


,player_name,distance
0,Robert Huth,2.492772
1,Kurt Happy Zouma,2.689903
2,Gary Cahill,2.727683
3,Micah Richards,2.842840
4,Sebastian Prödl,2.886859


PCA + NearestNeighbors:


,player_name,distance
1,Gary Cahill,2.278242
2,Robert Huth,2.403701
3,Kurt Happy Zouma,2.500897
4,Micah Richards,2.711384
5,Sebastian Prödl,2.826680


Top-5 overlap: 100%



### Model C — UMAP + NearestNeighbors

- Standardize player features using `StandardScaler`.
- Apply UMAP to learn a nonlinear representation of player profiles.
- Reduce the feature space to 5 dimensions.
- Use `NearestNeighbors` with Euclidean distance.
- Return the K nearest players in the UMAP embedding.

**Goal:** Evaluate whether nonlinear dimensionality reduction improves player similarity compared with the baseline and PCA.


In [33]:

import umap
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from src.similarity import find_similar_players

# Select Wes Morgan and centre-back features
player_id = 3813
data = matrices["centre_back"]
X = data["X"]

# Learn a nonlinear player representation
umap_model = umap.UMAP(
    n_neighbors=10,
    n_components=5,
    min_dist=0.1,
    metric="euclidean",
    random_state=42,
)

X_umap = umap_model.fit_transform(X)

print("Original dimensions:", X.shape[1])
print("UMAP dimensions:", X_umap.shape[1])

# Fit nearest-neighbor search
model = NearestNeighbors(metric="euclidean")
model.fit(X_umap)

# Find the nearest players to Wes Morgan
position = X.index.get_loc(player_id)

distances, indices = model.kneighbors(
    X_umap[position:position + 1],
    n_neighbors=6,
)

# Build recommendations
umap_neighbors = pd.DataFrame({
    "player_id": X.index[indices[0]],
    "distance": distances[0],
})

# Exclude Wes Morgan
umap_neighbors = umap_neighbors[
    umap_neighbors["player_id"] != player_id
].head(5).copy()

# Add player names
umap_neighbors["player_name"] = data["players"].loc[
    umap_neighbors["player_id"], "player_name"
].to_numpy()

# Compare with baseline
baseline = find_similar_players(player_id, matrices, top_k=5)

print("\nModel A — Baseline")
display(baseline[["player_name", "distance"]])

print("\nModel C — UMAP + NearestNeighbors")
display(umap_neighbors[["player_name", "distance"]])

overlap = len(
    set(baseline["player_id"])
    & set(umap_neighbors["player_id"])
) / 5

print(f"Top-5 overlap: {overlap:.0%}")


Original dimensions: 11
UMAP dimensions: 5

Model A — Baseline


/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,player_name,distance
0,Robert Huth,2.492772
1,Kurt Happy Zouma,2.689903
2,Gary Cahill,2.727683
3,Micah Richards,2.842840
4,Sebastian Prödl,2.886859



Model C — UMAP + NearestNeighbors


,player_name,distance
1,Robert Huth,0.399220
2,Gary Cahill,0.428870
3,Kurt Happy Zouma,0.576512
4,Sébastien Aymar Bassong Nguena,0.717543
5,John Terry,0.817148


Top-5 overlap: 60%


In [38]:
import importlib
import src.similarity as similarity

importlib.reload(similarity)

print(similarity.__file__)
print("UMAP available:", hasattr(similarity, "build_umap_models"))

/Users/bilal/Desktop/football-player-similarity/src/similarity.py
UMAP available: True


In [39]:
from src.similarity import (
    build_role_matrices,
    find_similar_players,
    build_umap_models,
    find_similar_players_umap,
)

# Build role-specific matrices
matrices = build_role_matrices()

# Test Euclidean similarity first
results = find_similar_players(
    player_id=3813,
    competition_id=2,
    matrices=matrices,
    top_k=10,
)

display(results)

# Build UMAP representations
models = build_umap_models(matrices)

# Find neighbors across leagues
results_umap = find_similar_players_umap(
    player_id=3813,
    competition_id=2,
    matrices=matrices,
    models=models,
    top_k=10,
)

display(results_umap)


,competition_id,season_id,player_id,player_name,minutes_played,distance
0,2,27,40123,Robert Huth,3313.316667,2.460923
1,9,27,8313,Kyriakos Papadopoulos,1032.666667,2.513989
2,12,27,7052,Alessandro Gamberini,1875.666667,2.542010
3,2,27,3456,Kurt Happy Zouma,2019.316667,2.614859
4,7,27,26481,Sébastien Squillaci,2677.633333,2.616515
5,12,27,40350,Siniša Anđelković,2009.650000,2.673325
6,2,27,3645,Gary Cahill,2064.816667,2.706028
7,7,27,40527,Brou Benjamin Angoua,1311.466667,2.707556
8,2,27,3309,Sebastian Prödl,1864.550000,2.709420
9,2,27,42700,Micah Richards,2143.466667,2.816471


/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random

,competition_id,season_id,player_id,player_name,minutes_played,distance
0,12,27,40426,Guglielmo Stendardo,1175.600000,0.351127
1,9,27,8313,Kyriakos Papadopoulos,1032.666667,0.361496
2,7,27,40527,Brou Benjamin Angoua,1311.466667,0.474953
3,12,27,40350,Siniša Anđelković,2009.650000,0.486252
4,12,27,7966,Filip Helander,1955.333333,0.530010
5,12,27,45590,Nicolò Cherubin,1112.633333,0.575218
6,7,27,23203,Stopilla Sunzu,958.700000,0.625338
7,12,27,8278,Nicolás Federico Spolli,1152.983333,0.650658
8,12,27,9939,Maurício dos Santos Nascimento,2034.700000,0.651520
9,12,27,7052,Alessandro Gamberini,1875.666667,0.696819
